# 🤖 Roboshelf AI — GPU-gyorsított Humanoid RL tanítás

**Google Colab notebook a Fázis 1/2 PPO tanításhoz**

Ez a notebook:
- NVIDIA T4/A100 GPU-t használ a tanításhoz
- MuJoCo + Stable-Baselines3 PPO pipeline
- Humanoid-v5 környezet (Fázis 1) vagy MuJoCo Playground G1 (Fázis 2)
- Nagyobb batch méretekkel és gyorsabb tanítással mint a MacBook M2
- Eredmények letölthetők .zip-ben

⚡ **Runtime → Change runtime type → GPU (T4 vagy A100)**

In [ ]:
# === 1. GPU ELLENŐRZÉS ===
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], capture_output=True, text=True)
print('🖥️ GPU információ:')
print(f'  {result.stdout.strip()}')
print()

import torch
print(f'  PyTorch: {torch.__version__}')
print(f'  CUDA elérhető: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  CUDA eszköz: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# === 2. CSOMAGOK TELEPÍTÉSE ===
!pip install -q mujoco gymnasium stable-baselines3 tensorboard

import mujoco
import gymnasium as gym
import stable_baselines3
print(f'✅ MuJoCo: {mujoco.__version__}')
print(f'✅ Gymnasium: {gymnasium.__version__}')
print(f'✅ SB3: {stable_baselines3.__version__}')

In [ ]:
# === 3. TANÍTÁSI KONFIGURÁCIÓ ===
# Válassz szintet a cellát szerkesztve:

LEVEL = 'colab_gpu'  # @param ['teszt', 'kozepes', 'colab_gpu', 'colab_pro']

CONFIGS = {
    'teszt': {
        'total_timesteps': 100_000,
        'n_steps': 2048,
        'batch_size': 512,
        'n_epochs': 10,
        'n_envs': 8,
        'learning_rate': 3e-4,
        'clip_range': 0.2,
        'desc': 'Gyors teszt (~2 perc)',
    },
    'kozepes': {
        'total_timesteps': 5_000_000,
        'n_steps': 2048,
        'batch_size': 512,
        'n_epochs': 10,
        'n_envs': 8,
        'learning_rate': 1e-4,
        'clip_range': 0.15,
        'desc': 'Közepes (~15 perc)',
    },
    'colab_gpu': {
        'total_timesteps': 50_000_000,
        'n_steps': 2048,
        'batch_size': 1024,
        'n_epochs': 10,
        'n_envs': 16,
        'learning_rate': 1e-4,
        'clip_range': 0.15,
        'desc': 'T4 GPU teljes (~1-2 óra)',
    },
    'colab_pro': {
        'total_timesteps': 100_000_000,
        'n_steps': 4096,
        'batch_size': 2048,
        'n_epochs': 10,
        'n_envs': 32,
        'learning_rate': 1e-4,
        'clip_range': 0.1,
        'desc': 'A100 Pro teljes (~1-3 óra)',
    },
}

cfg = CONFIGS[LEVEL]
print(f'📋 Szint: {LEVEL} — {cfg["desc"]}')
print(f'   Timesteps: {cfg["total_timesteps"]:,}')
print(f'   Batch: {cfg["batch_size"]}, Envs: {cfg["n_envs"]}')
print(f'   LR: {cfg["learning_rate"]}, Clip: {cfg["clip_range"]}')

In [ ]:
# === 4. KÖRNYEZET LÉTREHOZÁSA ===
from stable_baselines3.common.vec_env import SubprocVecEnv, VecNormalize
from stable_baselines3.common.utils import set_random_seed

def make_env(rank, seed=42):
    def _init():
        env = gym.make('Humanoid-v5')
        env.reset(seed=seed + rank)
        return env
    return _init

set_random_seed(42)
env = SubprocVecEnv([make_env(i) for i in range(cfg['n_envs'])])
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.0, clip_reward=10.0)

print(f'✅ {cfg["n_envs"]}× Humanoid-v5 párhuzamos környezet létrehozva')
print(f'   Obs space: {env.observation_space.shape}')
print(f'   Action space: {env.action_space.shape}')

In [ ]:
# === 5. PPO MODELL ===
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
import os

# Könyvtárak
os.makedirs('roboshelf_results/models/best', exist_ok=True)
os.makedirs('roboshelf_results/logs', exist_ok=True)

policy_kwargs = dict(net_arch=dict(pi=[256, 256], vf=[256, 256]))

model = PPO(
    'MlpPolicy',
    env,
    policy_kwargs=policy_kwargs,
    learning_rate=cfg['learning_rate'],
    n_steps=cfg['n_steps'],
    batch_size=cfg['batch_size'],
    n_epochs=cfg['n_epochs'],
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=cfg['clip_range'],
    ent_coef=0.001,
    vf_coef=0.5,
    max_grad_norm=0.5,
    tensorboard_log='roboshelf_results/logs',
    verbose=1,
    seed=42,
    device='cuda',  # GPU!
)

print(f'✅ PPO modell létrehozva (device: cuda)')
print(f'   Hálózat: pi=[256,256], vf=[256,256]')

In [ ]:
# === 6. EVAL CALLBACK ===
from stable_baselines3.common.vec_env import DummyVecEnv

eval_env = DummyVecEnv([lambda: gym.make('Humanoid-v5')])
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, training=False)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path='roboshelf_results/models/best',
    log_path='roboshelf_results/logs',
    eval_freq=max(cfg['total_timesteps'] // 20, 5000),
    n_eval_episodes=5,
    deterministic=True,
)

checkpoint_callback = CheckpointCallback(
    save_freq=max(cfg['total_timesteps'] // 10, 10000),
    save_path='roboshelf_results/models/checkpoints',
    name_prefix=f'humanoid_ppo_{LEVEL}_colab',
)

print('✅ Eval és checkpoint callbackek beállítva')

In [ ]:
# === 7. TANÍTÁS INDÍTÁSA ===
import time

print(f'🚀 Tanítás indítása: {cfg["total_timesteps"]:,} timestep, {cfg["n_envs"]}× env, GPU')
print(f'   Szint: {LEVEL} — {cfg["desc"]}')
print('=' * 60)

start = time.time()

model.learn(
    total_timesteps=cfg['total_timesteps'],
    callback=[eval_callback, checkpoint_callback],
    tb_log_name=f'roboshelf_{LEVEL}_colab',
    progress_bar=True,
)

elapsed = time.time() - start
print(f'\n⏱️  Tanítás befejezve: {elapsed/60:.1f} perc ({elapsed/3600:.1f} óra)')

In [ ]:
# === 8. MENTÉS ===
model.save('roboshelf_results/models/humanoid_ppo_colab_final')
env.save('roboshelf_results/models/humanoid_ppo_colab_vecnormalize.pkl')
print('💾 Modell és VecNormalize mentve')

# Best modell mellé is VecNormalize
env.save('roboshelf_results/models/best/best_model_vecnormalize.pkl')
print('💾 Best VecNormalize mentve')

In [ ]:
# === 9. KIÉRTÉKELÉS ===
import numpy as np

# Eval env a helyes VecNormalize-zel
eval_env2 = DummyVecEnv([lambda: gym.make('Humanoid-v5')])
eval_env2 = VecNormalize.load('roboshelf_results/models/humanoid_ppo_colab_vecnormalize.pkl', eval_env2)
eval_env2.training = False
eval_env2.norm_reward = False

rewards = []
lengths = []
for ep in range(10):
    obs = eval_env2.reset()
    done = False
    total_reward = 0
    steps = 0
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = eval_env2.step(action)
        total_reward += reward[0]
        steps += 1
    rewards.append(total_reward)
    lengths.append(steps)
    print(f'  Epizód {ep+1}: reward={total_reward:.1f}, hossz={steps}')

print(f'\n📈 Átlag reward: {np.mean(rewards):.1f} (±{np.std(rewards):.1f})')
print(f'📏 Átlag hossz: {np.mean(lengths):.0f}')
print(f'🏆 Legjobb: {np.max(rewards):.1f}')

In [ ]:
# === 10. LETÖLTÉS ===
!cd roboshelf_results && zip -r ../roboshelf_colab_results.zip models/ logs/

from google.colab import files
files.download('roboshelf_colab_results.zip')
print('\n📦 Eredmények letöltése...')
print('   A MacBook-on kicsomagolás:')
print('   unzip roboshelf_colab_results.zip -d ~/Documents/roboshelf-ai-dev/roboshelf-results/')

In [ ]:
# === OPCIONÁLIS: TENSORBOARD ===
%load_ext tensorboard
%tensorboard --logdir roboshelf_results/logs